In [ ]:
# document structure
from langchain_core.documents import Document

In [ ]:
doc = Document(
    page_content="This is the main text content I am using to create RAG",
    metadata={
        "source": "example.txt",
        "pages": "1",
        "author": "gaurav",
        "date_created": "2026-05-27"
    }
)
doc

In [ ]:
# Textloader
from langchain_community.document_loaders import TextLoader

loader=TextLoader("./data/text_files/text.txt", encoding="utf-8")
document=loader.load()
print(document)

In [ ]:
# Directory loader
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader

dir_loader=DirectoryLoader(
    "./data/pdfs",
    glob="**/*.pdf",
    loader_cls=PyMuPDFLoader,
    # loader_kwargs={"encoding":"utf-8"},
    show_progress=False
)
documents=dir_loader.load()
print(documents)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs=text_splitter.split_documents(documents)
    print(f"split {len(documents)} documents into {len(split_docs)} chunks")

    # show example:
    if split_docs:
      print(f"\nExample chunk")
      print(f"Content: {split_docs[0].page_content[:200]}...")
      print(f"metadata: {split_docs[0].metadata}")

    return split_docs


In [ ]:
chunks=split_documents(documents)
chunks

### embeddings and vectorstoreDB

In [ ]:
import numpy as np
import os
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
class EmbeddingModel:
    def __init__(self, model_name:str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self.load_model()

    def load_model(self):
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e: 
            print(f"Error loading model {self.model_name}: {e}")
            raise
    
    def generate_embedding(self, texts:List[str]) -> np.ndarray:
        if not self.model:
            raise ValueError("Model not loaded")
        try:
            embedding = self.model.encode(texts, show_progress_bar=True)
            return embedding
        except Exception as e:
            print(f"Error generating embedding for text: {e}")
            raise

embedding_model = EmbeddingModel()

### vectorStore

In [ ]:
class vectorStore:
    def __init__(self, collection_name:str = "pdf_documents", persist_directory:str = "./data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        try:
            # Create Persistent ChromaDB Client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or Create Collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )

            print(f"Vector Store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in Collection: {self.collection.count()}")
        
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match numser of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        document_text = []
        embedding_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            document_text.append(doc.page_content)

            # Embedding
            embedding_list.append(embedding.tolist())

        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embedding_list,
                metadatas=metadatas,
                documents=document_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

In [ ]:
vectorstore = vectorStore()
vectorstore

In [ ]:
## Convert the text to embeddings
texts = [doc.page_content for doc in chunks]
## Generate the embeddings
embeddings = embedding_model.generate_embedding(texts)

## Store in the vector DB
vectorstore.add_documents(chunks, embeddings)

In [ ]:
class RAGretriever:
    def __init__(self, vectorstore: vectorStore, embedding_model: EmbeddingModel):
        self.vector_store = vectorstore
        self.embedding_model = embedding_model

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        print(f"Retrieving relevant documents for query: '{query}'")
        print(f"Top K: {top_k}, Score Threshold: {score_threshold}")

        # Generate embedding for the query
        query_embedding = self.embedding_model.generate_embedding([query])[0]

        # search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k,
                include=["documents", "metadatas", "distances"]
            )
            
            retrieved_docs = []
            if results and "documents" in results and results["documents"] and len(results["documents"]) > 0:
                documents = results["documents"][0]
                metadatas = results["metadatas"][0]
                distances = results["distances"][0]
                ids = results["ids"][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    similarity_score = 1 - distance
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "document": document,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "distance": distance,
                            "rank": i + 1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents above the similarity threshold")
            else:
                print("No documents retrieved from vector store")
            return retrieved_docs
        except Exception as e:
            print(f"Error retrieving documents: {e}")
            import traceback
            traceback.print_exc()
            raise

retriever = RAGretriever(vectorstore, embedding_model)

In [ ]:
retrieved_docs = retriever.retrieve("who was John Galt")

In [ ]:
# giving prompt to the llm ollama
class LLMClient:
    def __init__(self, model_name:str = "gemma:2b", api_url:str = "http://localhost:11434/api/generate"):
        self.model_name = model_name
        self.api_url = api_url

    def generate_response(self, prompt:str, max_tokens:int = 512) -> str:
        import requests
        try:
            response = requests.post(
                self.api_url,
                json={
                    "model": self.model_name,
                    "prompt": prompt,
                    "stream": False
                },
                timeout=60
            )
            response.raise_for_status()
            data = response.json()
            if 'response' in data:
                return data['response'].strip()
            else:
                print("No response returned from LLM")
                return ""
        except requests.exceptions.ConnectionError as e:
            print(f"Connection Error: Could not connect to Ollama at {self.api_url}")
            print(f"Make sure Ollama is running: ollama serve")
            raise
        except Exception as e:
            print(f"Error generating response from LLM: {e}")
            raise

llm_client = LLMClient()

In [ ]:
prompt = f"Based on the retrieved documents{retrieved_docs}, who is John Galt? Provide a concise answer based on the information in the documents."
response = llm_client.generate_response(prompt)
print(f"LLM Response: {response}")

In [ ]:
def AskLLM(question:str):
    # retrieved_docs = retriever.retrieve(question)
    prompt = f"Based on the retrieved documents{retrieved_docs}, answer the question: {question}"
    response = llm_client.generate_response(prompt)
    return response

In [ ]:
AskLLM("who was John Galt")